# QA — Dockerize the Service

**Duration:** 5 min

## Dockerfile

In [ ]:
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY model.pkl .
COPY api/ api/

EXPOSE 8000
CMD ["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8000"]

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/mlops-real-world/mlops-qa-docker.ipynb)

In [ ]:
fastapi==0.111.0
uvicorn==0.30.1
scikit-learn==1.5.0
pandas==2.2.2
numpy==1.26.4
joblib==1.4.2
pydantic==2.7.1

## Build & Run

In [ ]:
docker build -t churn-api:latest .
docker run -p 8000:8000 churn-api:latest

# Test it
curl http://localhost:8000/health

```
{"status": "ok", "model_features": 19}
```

## Integration Tests

In [ ]:
import pytest
from fastapi.testclient import TestClient
from api.app import app

client = TestClient(app)

def test_health():
    r = client.get('/health')
    assert r.status_code == 200
    assert r.json()['status'] == 'ok'

def test_predict_returns_probability():
    payload = {'tenure': 12, 'MonthlyCharges': 65.0, 'TotalCharges': 780.0,
               'Contract': 0, 'InternetService': 1}
    r = client.post('/predict', json=payload)
    assert r.status_code == 200
    data = r.json()
    assert 0 <= data['churn_probability'] <= 1
    assert data['risk'] in ['low','medium','high']